# imports

create the labeled base dataset

In [1]:
import os, glob
import pandas as pd
import numpy as np
import re

In [2]:
DATA_ROOT = "/Users/ferolahima/Desktop/gazepoint_project/data"  # data/<Video Name>/fixations/*.csv
TASKS_CSV = "/Users/ferolahima/Desktop/gazepoint_project/Export-Video Mappings.xlsx - Foglio1.csv"
OUTPUT_CSV = "outputs/all_fixations_with_taskid.csv"

USER_LABELS = ["User 0", "User 1", "User 2", "User 3", "User 4", "User 5", "User 6", "User 7"]

In [3]:
tasks_all = pd.read_csv(TASKS_CSV, sep=None, engine="python")
print("Times CSV columns:", list(tasks_all.columns))
print("Times CSV rows:", len(tasks_all))

needed = ["Video Name", "User ID", "Task Id", "Starting time in seconds", "Finishing time in seconds"]

def _normalize_col(name):
    if name is None:
        return ""
    return "".join(ch for ch in str(name).strip().lower() if ch.isalnum())

existing_cols = { _normalize_col(col): col for col in tasks_all.columns if isinstance(col, str) and _normalize_col(col) }
rename_map = {}
missing = []

for target in needed:
    key = _normalize_col(target)
    actual = existing_cols.get(key)
    if actual:
        rename_map[actual] = target
    else:
        missing.append(target)

if missing:
    available = [col for col in tasks_all.columns if isinstance(col, str)]
    raise KeyError(f"Times CSV missing columns: {missing}. Available columns: {available}")

tasks_all = tasks_all.rename(columns=rename_map)
tasks_all = tasks_all[needed].copy()
tasks_all["Starting time in seconds"]  = pd.to_numeric(tasks_all["Starting time in seconds"], errors="coerce")
tasks_all["Finishing time in seconds"] = pd.to_numeric(tasks_all["Finishing time in seconds"], errors="coerce")

print("\nSample of times rows:")
display(tasks_all.head(15) if hasattr(__builtins__, 'display') else print(tasks_all.head(5)))


Times CSV columns: ['Unnamed: 0', 'Video Name', 'User ID', 'Task Id', 'Starting time (minute:second)', 'Starting time in seconds', 'Finishing time (minute:second)', 'Finishing time in seconds', 'Duration in seconds']
Times CSV rows: 809

Sample of times rows:


,Video Name,User ID,Task Id,Starting time in seconds,Finishing time in seconds
0,P3_S2_ENG,User 0,1.1,81,129
1,P3_S2_ENG,User 0,3.3,432,459
2,P3_S2_ENG,User 0,2.3,647,659
3,P3_S2_ENG,User 0,1.2,922,933
4,P3_S2_ENG,User 0,3.1,1015,1018
5,P3_S2_ENG,User 0,2.1,1083,1092
6,P3_S2_ENG,User 0,1.5,1148,1189
7,P3_S2_ENG,User 1,3.6,16,23
8,P3_S2_ENG,User 1,2.7,74,81
9,P3_S2_ENG,User 1,1.8,126,176


In [4]:
if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}")

participant_folders = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))]
participant_folders.sort()
print(f"\nFound {len(participant_folders)} participant folders under '{DATA_ROOT}':")
for name in participant_folders:
    print("  -", name)


Found 27 participant folders under '/Users/ferolahima/Desktop/gazepoint_project/data':
  - P10_S1_ITA
  - P11_S2_ITA
  - P12_S1_ENG
  - P13_S2ENG
  - P14_S1_Ita
  - P15_S2_Ita
  - P16_S1_Ita
  - P17_S2_Ita
  - P18_S1_Ita
  - P19_S2_Ita
  - P20_S1_Ita
  - P21_S2_iTA
  - P22_S1_Eng
  - P23_S1_Ita
  - P27_S1_Ita
  - P28_S2_Ita
  - P29_S2_Ita
  - P30_S2_Ita
  - P31_S2_eng
  - P32_S1_Ita
  - P3_S2_ENG
  - P4_Sequence1_Italiano
  - P5_Sequence1_it
  - P6_S1IT
  - P7_ENG
  - P8_S1_ITA
  - P9_S2_ITA


In [5]:
def find_user_files_all(participant_folder_name: str) -> dict:
    p_path = os.path.join(DATA_ROOT, participant_folder_name)
    found = {label: {"fixation": [], "gaze": []} for label in USER_LABELS}

    for fpath in glob.iglob(os.path.join(p_path, "**", "*.csv"), recursive=True):
        fname = os.path.basename(fpath)
        user_hit = next((label for label in USER_LABELS if label in fname), None)
        if not user_hit:
            continue  # skip files not tied to a user label
        kind = "fixation" if "fixation" in fname.lower() else "gaze"
        found[user_hit][kind].append(fpath)

    # prune users with no files
    return {u: kinds for u, kinds in found.items() if kinds["fixation"] or kinds["gaze"]}

# --- Discover files for all participants ---
user_files_all_map = {}
print("\nFiles discovered (fixations + gaze):")
for pname in participant_folders:
    mapping = find_user_files_all(pname)
    user_files_all_map[pname] = mapping
    if not mapping:
        print(f"  {pname}: — none —")
    else:
        print(f"  {pname}:")
        for user, kinds in mapping.items():
            print(f"    {user}: {len(kinds['fixation'])} fixation, {len(kinds['gaze'])} gaze")


Files discovered (fixations + gaze):
  P10_S1_ITA:
    User 0: 1 fixation, 1 gaze
  P11_S2_ITA:
    User 0: 1 fixation, 1 gaze
  P12_S1_ENG:
    User 0: 1 fixation, 1 gaze
    User 1: 1 fixation, 1 gaze
  P13_S2ENG:
    User 0: 1 fixation, 1 gaze
  P14_S1_Ita:
    User 0: 1 fixation, 1 gaze
    User 1: 1 fixation, 1 gaze
  P15_S2_Ita:
    User 0: 1 fixation, 1 gaze
  P16_S1_Ita:
    User 0: 1 fixation, 1 gaze
  P17_S2_Ita:
    User 0: 1 fixation, 1 gaze
  P18_S1_Ita:
    User 0: 1 fixation, 1 gaze
  P19_S2_Ita:
    User 0: 1 fixation, 1 gaze
  P20_S1_Ita:
    User 0: 1 fixation, 1 gaze
  P21_S2_iTA:
    User 0: 1 fixation, 1 gaze
  P22_S1_Eng:
    User 0: 1 fixation, 1 gaze
  P23_S1_Ita:
    User 0: 1 fixation, 1 gaze
    User 1: 1 fixation, 1 gaze
  P27_S1_Ita:
    User 0: 1 fixation, 1 gaze
  P28_S2_Ita:
    User 0: 1 fixation, 1 gaze
  P29_S2_Ita:
    User 0: 1 fixation, 1 gaze
  P30_S2_Ita:
    User 0: 1 fixation, 1 gaze
  P31_S2_eng:
    User 0: 1 fixation, 1 gaze
  P32_S1_Ita:
 

In [6]:
def match_user_file_by_time(
    video_name: str,
    user_id: str,
    csv_path: str,
    tasks_table: pd.DataFrame,
    source: str = "auto",
) -> pd.DataFrame:

    # 1) Filter task windows for this participant + user
    t = tasks_table[
        (tasks_table["Video Name"] == video_name) &
        (tasks_table["User ID"] == user_id)
    ].copy()

    # If there are no tasks for this (video_name, user_id), still return the file with NaN Task Ids
    t["Starting time in seconds"]  = pd.to_numeric(t["Starting time in seconds"], errors="coerce")
    t["Finishing time in seconds"] = pd.to_numeric(t["Finishing time in seconds"], errors="coerce")

    # 2) Load CSV
    df = pd.read_csv(csv_path, sep=None, engine="python")

    # 3) Normalize TIME(...) → TIME (only the first occurrence kept)
    time_cols = [c for c in df.columns if c.startswith("TIME(")]
    if time_cols:
        df = df.rename(columns={time_cols[0]: "TIME"})
        # if more TIME(...) columns exist, drop them
        for extra in time_cols[1:]:
            df = df.drop(columns=extra, errors="ignore")

    # make sure TIME is numeric (float seconds with decimals)
    df["TIME"] = pd.to_numeric(df["TIME"], errors="coerce")

    # 4) Assign Task Id by interval membership
    df["Task Id"] = pd.NA
    for _, row in t.iterrows():
        start = row["Starting time in seconds"]
        end   = row["Finishing time in seconds"]
        if pd.isna(start) or pd.isna(end):
            continue
        mask = (df["TIME"] >= start) & (df["TIME"] <= end)  
        df.loc[mask, "Task Id"] = row["Task Id"]

    # 5) Context
    df.insert(0, "User ID (file)", user_id)
    df.insert(0, "Video Name (folder)", video_name)

    return df


In [13]:
# === BUILD SEPARATE TABLES: FIXATIONS vs GAZE ===
all_fixations = []
all_gaze = []

for video_name in participant_folders:
    u_map = user_files_all_map.get(video_name, {})  # from find_user_files_all(...)
    if not u_map:
        continue

    for user_id, kinds in u_map.items():
        # Fixation files
        for fx_path in kinds.get("fixation", []):
            df_fx = match_user_file_by_time(video_name, user_id, fx_path, tasks_all, source="fixation")
            all_fixations.append(df_fx)

        # Raw gaze files
        for gz_path in kinds.get("gaze", []):
            df_gz = match_user_file_by_time(video_name, user_id, gz_path, tasks_all, source="gaze")
            all_gaze.append(df_gz)

# Concatenate separately
fixations_df = pd.concat(all_fixations, ignore_index=True) if all_fixations else pd.DataFrame()
gaze_df      = pd.concat(all_gaze, ignore_index=True) if all_gaze else pd.DataFrame()

# filter some columns from fixations and gaze
cols_to_drop = [
    "Unnamed: 21",
    "MEDIA_NAME",
    "BPOGX",
    "BPOGY",
    "BPOGV",
]

fixations_df = fixations_df.drop(columns=cols_to_drop, errors="ignore")
gaze_df = gaze_df.drop(columns=cols_to_drop, errors="ignore")

/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_76971/2815081505.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fixations_df = pd.concat(all_fixations, ignore_index=True) if all_fixations else pd.DataFrame()


In [14]:
task_mapping = {
    '1.1': 'C011', '1.2': 'C100', '1.3': 'C000', '1.4': 'Cnull', '1.5': 'C100',
    '1.6': 'C110', '1.7': 'C011', '1.8': 'C001', '1.9': 'C111', '2.1': 'B000',
    '2.2': 'Bnull', '2.3': 'B112', '2.4': 'B113', '2.5': 'B012', '2.6': 'B013',
    '2.7': 'B002', '2.8': 'B102', '2.9': 'B103', '3.1': 'A000', '3.2': 'A100',
    '3.3': 'A010', '3.4': 'A101', '3.5': 'A001', '3.6': 'A111', '3.7': 'A110',
    '3.8': 'A011', '3.9': 'Anull'
}

animation_mapping = {
    'A': {
        'type': 'FADE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
    'B': {
        'type': 'BOUNCE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': {0: 'Left to Right', 1: 'Top to Bottom', 2: 'Bouncing', 3: 'Pulsing'},
    },
    'C': {
        'type': 'SCROLL',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
}

In [15]:
def decode_task(task_id) -> dict:
    tid = str(task_id)
    code = task_mapping.get(tid, "")
    if not code:
        return {"type": "Unknown", "delay": None, "ease": None, "direction": None, "code": None}

    family = code[0]
    fam_info = animation_mapping.get(family, {})
    result = {"type": fam_info.get("type", "Unknown"), "delay": None, "ease": None, "direction": None, "code": code}

    # handle null cases like Anull / Bnull / Cnull
    if "null" in code.lower():
        return result

    # extract up to three digit indices after the family letter
    digits = [d for d in code[1:] if d.isdigit()]
    while len(digits) < 3:
        digits.append("0")
    idx_delay, idx_ease, idx_dir = (int(digits[0]), int(digits[1]), int(digits[2]))

    def pick(options, i):
        """options can be list or dict; return None/UNSPEC if not available."""
        if options is None:
            return None
        if isinstance(options, dict):
            return options.get(i, f"UNSPEC({i})")
        if isinstance(options, list):
            return options[i] if 0 <= i < len(options) else f"UNSPEC({i})"
        return None

    result["delay"]     = pick(fam_info.get("delay"),     idx_delay)
    result["ease"]      = pick(fam_info.get("ease"),      idx_ease)
    result["direction"] = pick(fam_info.get("direction"), idx_dir)
    result["type"] = fam_info.get("type", "Unknown")

    return result

In [16]:
# data with animations details
fixations_df["delay"]     = fixations_df["Task Id"].apply(lambda tid: decode_task(tid)["delay"])
fixations_df["ease"]      = fixations_df["Task Id"].apply(lambda tid: decode_task(tid)["ease"])
fixations_df["direction"] = fixations_df["Task Id"].apply(lambda tid: decode_task(tid)["direction"])
fixations_df["type"]      = fixations_df["Task Id"].apply(lambda tid: decode_task(tid)["type"])

gaze_df["delay"]     = gaze_df["Task Id"].apply(lambda tid: decode_task(tid)["delay"])
gaze_df["ease"]      = gaze_df["Task Id"].apply(lambda tid: decode_task(tid)["ease"])
gaze_df["direction"] = gaze_df["Task Id"].apply (lambda tid: decode_task(tid)["direction"])
gaze_df["type"]      = gaze_df["Task Id"].apply(lambda tid: decode_task(tid)["type"])


In [17]:
# All rows
fixations_df.to_csv("outputs/fixations_all_rows.csv", index=False)
gaze_df.to_csv("outputs/gaze_all_rows.csv", index=False)

# Only rows within tasks
fx_tasks = fixations_df.dropna(subset=["Task Id"]).copy()
gz_tasks = gaze_df.dropna(subset=["Task Id"]).copy()
fx_tasks.to_csv("outputs/fixations_within_tasks.csv", index=False)
gz_tasks.to_csv("outputs/gaze_within_tasks.csv", index=False)

print("Fixations:", fixations_df.shape, "→ within tasks:", fx_tasks.shape)
print("Gaze     :", gaze_df.shape, "→ within tasks:", gz_tasks.shape)

# Quick peek
print("\nFixations sample:")
display(fixations_df.head(5) if hasattr(__builtins__, 'display') else fixations_df.head(5))
print("\nGaze sample:")
display(gaze_df.head(5) if hasattr(__builtins__, 'display') else gaze_df.head(5))

Fixations: (100800, 24) → within tasks: (30816, 24)
Gaze     : (3310011, 24) → within tasks: (884836, 24)

Fixations sample:


,Video Name (folder),User ID (file),TIME,TIMETICK(f=10000000),FPOGX,FPOGY,FPOGS,FPOGD,FPOGID,FPOGV,...,RPMM,RPMMV,AOI,SACCADE_MAG,SACCADE_DIR,Task Id,delay,ease,direction,type
0,P10_S1_ITA,User 0,0.22998,90758379745,-0.02472,1.53958,0.01660,0.21338,2,1,...,2.88307,1,NaN,0.00000,0.00000,<NA>,None,None,None,Unknown
1,P10_S1_ITA,User 0,0.52588,90761337469,-0.07156,1.61755,0.24658,0.27930,3,1,...,2.83958,1,NaN,246.40483,223.11699,<NA>,None,None,None,Unknown
2,P10_S1_ITA,User 0,0.77197,90763801878,-0.02402,1.64018,0.54199,0.22998,4,1,...,2.83607,1,NaN,188.98454,345.01001,<NA>,None,None,None,Unknown
3,P10_S1_ITA,User 0,1.00195,90766100929,-0.04514,1.67480,0.78857,0.21338,5,1,...,2.96685,1,NaN,110.31447,222.67772,<NA>,None,None,None,Unknown
4,P10_S1_ITA,User 0,1.41309,90770209004,-0.37341,2.38828,1.21582,0.19727,6,1,...,3.06466,1,NaN,1990.99048,230.71854,<NA>,None,None,None,Unknown



Gaze sample:


,Video Name (folder),User ID (file),TIME,TIMETICK(f=10000000),FPOGX,FPOGY,FPOGS,FPOGD,FPOGID,FPOGV,...,RPMM,RPMMV,AOI,SACCADE_MAG,SACCADE_DIR,Task Id,delay,ease,direction,type
0,P10_S1_ITA,User 0,0.00000,90756080037,0.00000,0.00000,0.0000,0.00000,1,0,...,2.91313,0,NaN,0.0,0.0,<NA>,None,None,None,Unknown
1,P10_S1_ITA,User 0,0.01660,90756246206,0.03940,1.41659,0.0000,0.00000,1,0,...,2.87747,1,NaN,0.0,0.0,<NA>,None,None,None,Unknown
2,P10_S1_ITA,User 0,0.03271,90756408356,0.03685,1.41747,0.0000,0.00000,1,0,...,2.87113,1,NaN,0.0,0.0,<NA>,None,None,None,Unknown
3,P10_S1_ITA,User 0,0.04932,90756573544,0.03600,1.41776,0.0000,0.00000,1,0,...,2.81402,1,NaN,0.0,0.0,<NA>,None,None,None,Unknown
4,P10_S1_ITA,User 0,0.06543,90756736773,0.01830,1.43621,0.0166,0.04883,2,1,...,2.79450,1,NaN,0.0,0.0,<NA>,None,None,None,Unknown
